# Deepfake Video Detection

## Project Goal
The objective of this project is to develop a deepfake detection pipeline across two modalities — **video** and **image** — that extracts features using convolutional neural networks, applies dimensionality reduction techniques such as PCA or LDA, and classifies content as real or fake using machine learning classifiers.

> **This notebook covers the video detection component** (Muaataz Ismaeel). The image detection component is implemented separately by Bilal Hasanov.

**Pipeline overview:**
1. Load pre-extracted frames from `real/` and `fake/` directories
2. Compare two CNN backbones as feature extractors: **MobileNetV2** vs **EfficientNetV2-S** (both frozen, ImageNet weights)
3. Aggregate frame features into a single video-level representation via mean pooling
4. Reduce dimensionality with PCA; visualize class separation with LDA
5. Train and compare Logistic Regression, SVM, and MLP classifiers
6. Evaluate the best model and run inference on new videos

## 1. Install Dependencies

In [ ]:
%pip install -q kagglehub opencv-python-headless tqdm scikit-learn joblib tensorflow pandas seaborn

## 2. Environment Setup

Auto-detects Kaggle vs Google Colab and sets paths accordingly.

In [ ]:
import os, sys
from pathlib import Path

# ── Detect runtime environment ──────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/input')
try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_KAGGLE:
    WORKING_DIR  = Path('/kaggle/working/deepfake_workspace')
    REPO_DIR     = Path('/kaggle/working')   # modules live directly in the working dir on Kaggle
elif ON_COLAB:
    WORKING_DIR  = Path('/content/deepfake_workspace')
    REPO_DIR     = Path('/content/repo')
else:
    WORKING_DIR  = Path('./deepfake_workspace')
    REPO_DIR     = Path('.')  # already in project root locally

WORKING_DIR.mkdir(parents=True, exist_ok=True)

# ── Clone repo on Colab so .py modules are importable ────────────────────
if ON_COLAB:
    GITHUB_REPO = 'https://github.com/MuaCodez30/Deep-Learning-Project.git'
    if not REPO_DIR.exists():
        os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
    sys.path.insert(0, str(REPO_DIR))
elif ON_KAGGLE:
    # Modules are uploaded as Kaggle dataset or placed in /kaggle/working
    sys.path.insert(0, str(REPO_DIR))

print(f'ON_KAGGLE={ON_KAGGLE}  ON_COLAB={ON_COLAB}')
print(f'WORKING_DIR: {WORKING_DIR}')
print(f'Module search path entry: {sys.path[0]}')

## 3. Imports

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm
import tensorflow as tf

# Project modules
from dataset  import find_data_directory, load_dataset_summary, preprocess_frame
from model    import build_mobilenetv2_extractor, build_efficientnetv2_extractor
from features import extract_all_video_features, build_video_feature
from train    import fit_pca, fit_lda, train_all_models
from evaluate import (print_report, plot_confusion_matrix, plot_pca_scatter,
                      plot_lda_scatter, plot_pca_variance, list_misclassified)
from predict  import predict_video_from_frames

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
RNG  = random.Random(SEED)

## 4. Configuration

In [ ]:
# ── Feature extraction ──────────────────────────────────────────────────
BATCH_SIZE           = 32    # Frames per CNN forward pass
MAX_FRAMES_PER_VIDEO = 60    # Cap per video to keep memory tractable
FRAME_SIZE           = (224, 224)  # Required input size for EfficientNetV2
POOLING              = 'max'       # Global pooling strategy
AGGREGATION          = 'mean'      # How per-frame features are collapsed to a video vector

# ── Classifier ──────────────────────────────────────────────────────────
MLP_HIDDEN     = (256, 128)
TEST_SIZE      = 0.2
RANDOM_STATE   = 42

# ── Dataset path ────────────────────────────────────────────────────────
if ON_KAGGLE:
    _raw_path = Path('/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames')
elif ON_COLAB:
    import kagglehub
    _raw_path = Path(kagglehub.dataset_download('adham7elmy/faceforencispp-extracted-frames'))
else:
    _raw_path = Path('./data')  # local path

DATASET_ROOT = find_data_directory(_raw_path)
assert DATASET_ROOT is not None, f'real/ and fake/ not found under {_raw_path}'
print(f'Dataset root: {DATASET_ROOT}')
print(f'Working dir : {WORKING_DIR}')

## 5. Dataset Loading

In [ ]:
summary, real_frames, fake_frames, real_video_groups, fake_video_groups = \
    load_dataset_summary(DATASET_ROOT)

print(summary.to_string(index=False))

In [ ]:
# Visualize a sample real and fake frame side-by-side
sample_real = real_frames[min(16001, len(real_frames)-1)] if real_frames else None
sample_fake = fake_frames[min(16001, len(fake_frames)-1)] if fake_frames else None

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, path, title in [(axes[0], sample_real, 'Real'), (axes[1], sample_fake, 'Fake')]:
    if path is None:
        ax.axis('off'); ax.set_title(f'{title} (not found)'); continue
    ax.imshow(preprocess_frame(path))
    ax.axis('off'); ax.set_title(title)
plt.tight_layout(); plt.show()

## 6. Feature Extraction

We compared two CNN backbones — **MobileNetV2** and **EfficientNetV2-S** — and selected EfficientNetV2-S for the full pipeline. Both are frozen with ImageNet weights and produce a **1,280-D vector per frame** via Global Max Pooling. EfficientNetV2-S was chosen for its deeper architecture and stronger performance on image recognition benchmarks compared to the lighter MobileNetV2.

Frames are aggregated across all sampled frames into a single video-level vector via mean pooling.

In [ ]:
# Build both backbones and compare their output dimensions
mobilenet_extractor    = build_mobilenetv2_extractor(pooling=POOLING)
efficientnet_extractor = build_efficientnetv2_extractor(pooling=POOLING)

print(f'MobileNetV2    feature dimension: {mobilenet_extractor.output_shape[1]}')
print(f'EfficientNetV2 feature dimension: {efficientnet_extractor.output_shape[1]}')

# EfficientNetV2-S is selected as the primary extractor for the full pipeline
feature_extractor = efficientnet_extractor
print(f'\nUsing: EfficientNetV2-S')

In [ ]:
X, y, video_ids = extract_all_video_features(
    frames_dir       = DATASET_ROOT,
    feature_extractor= feature_extractor,
    max_frames       = MAX_FRAMES_PER_VIDEO,
    batch_size       = BATCH_SIZE,
    frame_size       = FRAME_SIZE,
    aggregation      = AGGREGATION,
    rng              = RNG,
    cache_dir        = WORKING_DIR,   # skip re-extraction if .npy files already exist
)
print(f'Features shape : {X.shape}')
print(f'Labels shape   : {y.shape}')

## 7. Dimensionality Reduction

PCA retains 99% of variance and reduces the 1280-D feature space to a much smaller set of components. LDA is used for multi-class visualization only (not for classification).

In [ ]:
scaler_pca, pca, X_pca = fit_pca(X, n_components=0.99, save_dir=WORKING_DIR)
print(f'PCA reduced {X.shape[1]} → {X_pca.shape[1]} dimensions')
print(f'Explained variance retained: {pca.explained_variance_ratio_.sum():.4f}')

In [ ]:
plot_pca_variance(pca)
plot_pca_scatter(X_pca, y)

In [ ]:
# LDA — multi-class visualization (real + 5 manipulation methods)
from dataset import group_frames_by_video, find_frame_files
from sklearn.preprocessing import StandardScaler

FAKE_METHODS = {'deepfakes':1,'face2face':2,'faceswap':3,'faceshifter':4,'neuraltextures':5}
CLASS_NAMES  = {0:'real',1:'deepfakes',2:'face2face',3:'faceswap',4:'faceshifter',5:'neuraltextures'}
MAX_FRAMES_LDA = 10

def select_by_method(fake_groups, method):
    return [vid for vid, paths in fake_groups.items()
            if any(f'/{method}/' in str(p).lower() or f'\\{method}\\' in str(p).lower() for p in paths)]

video_samples = [(vid, 0, real_video_groups[vid]) for vid in real_video_groups]
for m, cid in FAKE_METHODS.items():
    for vid in select_by_method(fake_video_groups, m):
        video_samples.append((vid, cid, fake_video_groups[vid]))

X_lda_input, Y_lda = [], []
for vid, cls, frames in video_samples:
    feat = build_video_feature(frames, feature_extractor, MAX_FRAMES_LDA,
                               BATCH_SIZE, FRAME_SIZE, AGGREGATION, RNG)
    if feat is not None:
        X_lda_input.append(feat); Y_lda.append(cls)
X_lda_input = np.array(X_lda_input); Y_lda = np.array(Y_lda)

_, lda, X_lda = fit_lda(X_lda_input, Y_lda, save_dir=WORKING_DIR)
plot_lda_scatter(X_lda, Y_lda, CLASS_NAMES)

## 8. Model Training

Three classifiers are trained on PCA-reduced features. The best by F1 score is selected and saved as `final_model.pkl`.

In [ ]:
best_model, results, split = train_all_models(
    X_pca        = X_pca,
    y            = y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    mlp_hidden   = MLP_HIDDEN,
    save_dir     = WORKING_DIR,
)
print('\nAll results:')
for r in results:
    print(f"  {r['model']}: F1 = {r['f1']:.4f}")

## 9. Evaluation

In [ ]:
y_pred = best_model.predict(split['X_test_scaled'])
print_report(split['y_test'], y_pred)
plot_confusion_matrix(split['y_test'], y_pred)
list_misclassified(split['idx_test'], y_pred, split['y_test'], video_ids)

## 10. Inference on a New Video

Point `example_frames_dir` at any folder of frames to get a Real/Fake prediction.

In [ ]:
pca_inf        = joblib.load(WORKING_DIR / 'pca.pkl')
scaler_inf     = joblib.load(WORKING_DIR / 'scaler.pkl')
classifier_inf = joblib.load(WORKING_DIR / 'final_model.pkl')

# Automatically pick the first real video for a quick demo
example_frames_dir = None
if real_video_groups:
    first_vid   = next(iter(real_video_groups))
    first_paths = real_video_groups[first_vid]
    if first_paths:
        example_frames_dir = first_paths[0].parent

if example_frames_dir and example_frames_dir.exists():
    label, confidence = predict_video_from_frames(
        frames_dir       = example_frames_dir,
        feature_extractor= feature_extractor,
        pca_model        = pca_inf,
        scaler_model     = scaler_inf,
        classifier       = classifier_inf,
        max_frames       = MAX_FRAMES_PER_VIDEO,
        batch_size       = BATCH_SIZE,
        frame_size       = FRAME_SIZE,
        aggregation      = AGGREGATION,
        rng              = RNG,
    )
    print(f'Prediction : {label}')
    print(f'Confidence : {confidence:.4f}')
else:
    print('Set example_frames_dir to a directory of frames to run inference.')